# Hybrid Retrieval with Full-Text Search

全文搜索是一种通过匹配文本中的特定关键词或短语来检索文档的传统方法，其结果根据词频等因子计算出的相关性得分进行排序。虽然语义搜索在理解语义和上下文方面表现更优，但全文搜索在精确关键词匹配方面更具优势，因此可作为语义搜索的有效补充。构建检索增强生成（RAG）管道时，一种常见做法是先通过语义搜索和全文搜索检索文档，再通过重新排序过程对结果进行优化。
<img src="../RAG/Advanced_RAG/hybrid_and_rerank.png">
该方法(Milvus)将文本转换为稀疏向量，用于BM25评分。用户只需输入原始文本即可导入文档，无需手动计算稀疏向量。Milvus会自动生成并存储这些稀疏向量。在搜索文档时，用户只需指定文本搜索查询，Milvus便会内部计算BM25得分，并返回排序结果。

Milvus还支持混合检索，即结合全文搜索与基于密集向量的语义搜索。通过平衡关键词匹配与语义理解，通常可提升搜索质量，为用户提供更优的搜索结果。

> - 目前，全文搜索功能已在Milvus Standalone、Milvus Distributed和Zilliz Cloud中提供，但尚未在Milvus Lite中支持（该功能计划在未来版本中实现）。如需更多信息，请联系support@zilliz.com。

## Preparation
### Install Pymilvus

In [1]:
#! pip install pymilvus -U

In [2]:
import os

os.environ['NLTK_DATA'] = r'F:\Teewon\Milvue\model\nltk_data'

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TORCH_HOME'] = CUSTOM_CACHE

## Set Deepseek API Key

我们将使用 Deepseek 的模型来生成向量嵌入和响应。您需要将 API 密钥 Deepseek_API_KEY 作为环境变量进行准备。

In [3]:
import dotenv

dotenv.load_dotenv('../.env')

True

## Setup and Configuration
Import the necessary libraries

In [4]:
from typing import List
from openai import OpenAI
from pymilvus import MilvusClient,DataType,Function,FunctionType,AnnSearchRequest,RRFRanker

We'll use the MilvusClient to establish a connection to the Milvus server.

In [5]:
uri='http://localhost:19530'
collection_name='full_text_demo'
mc=MilvusClient(uri,collection_name)
print(mc.get_server_version())

3.0.0


## 全文搜索的集合配置
为全文搜索设置一个集合需要进行多个配置步骤。让我们逐一说明这些步骤。

### 文本分析配置
对于全文搜索，我们需要定义文本应如何被处理。分析器在全文搜索中起着关键作用，它们将句子分解为词元，并执行词形还原、去除停用词等词汇分析操作。在此处，我们只需定义一个分析器即可。

In [6]:
analyzer_params={"tokenizer":"jieba","filter":["lowercase"]}

- "tokenizer":"standard" 把一段文本（如一句话）切分成一个个独立的词元（token）
- "filter":["lowercase"] 把所有词元统一转为小写

For more concept details about analyzer, please refer to the [analyzer documentation](https://milvus.io/docs/zh/analyzer-overview.md).

### 集合模式与BM25函数
现在我们定义了包含主键、文本内容、稀疏向量（用于全文搜索）、密集向量（用于语义搜索）以及元数据的模式。同时，我们为全文搜索配置了BM25函数。

BM25函数可自动将文本内容转换为稀疏向量，使Milvus能够处理全文搜索的复杂性，而无需手动生成稀疏嵌入。

In [7]:
schema=MilvusClient.create_schema()
schema.add_field(
    field_name='id',
    datatype=DataType.VARCHAR,
    is_primary=True,
    auto_id=True,
    max_length=100,
)
schema.add_field(
    field_name='content',
    datatype=DataType.VARCHAR,
    max_length=65535,
    analyzer_params=analyzer_params,
    enable_match=True,  # Enable text matching
    enable_analyzer=True,     # Enable text analysis
)
schema.add_field(
    field_name='sparse_vector',
    datatype=DataType.SPARSE_FLOAT_VECTOR,
)
schema.add_field(
    field_name='dense_vector',
    datatype=DataType.FLOAT_VECTOR,
    dim=4096    # Dimension for text-embedding-3-small
)
schema.add_field(
    field_name='metadata',
    datatype=DataType.JSON
)

bm25_function=Function(
    name='bm25',
    function_type= FunctionType.BM25,
    input_field_names=["content"],
    output_field_names="sparse_vector",
)

schema.add_function(bm25_function)

{'auto_id': True, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 100}, 'is_primary': True, 'auto_id': True}, {'name': 'content', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 65535, 'enable_match': True, 'enable_analyzer': True, 'analyzer_params': '{"tokenizer":"jieba","filter":["lowercase"]}'}}, {'name': 'sparse_vector', 'description': '', 'type': <DataType.SPARSE_FLOAT_VECTOR: 104>, 'is_function_output': True}, {'name': 'dense_vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 4096}}, {'name': 'metadata', 'description': '', 'type': <DataType.JSON: 23>}], 'enable_dynamic_field': False, 'enable_namespace': False, 'functions': [{'name': 'bm25', 'description': '', 'type': <FunctionType.BM25: 1>, 'input_field_names': ['content'], 'output_field_names': ['sparse_vector'], 'params': {}}]}

### 索引与集合创建
为了优化搜索性能，我们为稀疏和密集向量字段创建索引，然后在 Milvus 中创建集合。

In [8]:
index_params=MilvusClient.prepare_index_params()
index_params.add_index(
    field_name='sparse_vector',
    index_type="SPARSE_INVERTED_INDEX",
    metric_type= "BM25"
)
index_params.add_index(
    field_name='dense_vector',
    index_type="FLAT",
    metric_type= "IP"
)

if mc.has_collection(collection_name):
    mc.drop_collection(collection_name)

mc.create_collection(
    collection_name=collection_name,
    schema=schema,
    index_params=index_params,
)
print(f"Collection {collection_name} created successfully")

Collection full_text_demo created successfully


### 插入数据
在设置好集合后，我们通过准备包含文本内容及其向量表示的实体来插入数据。首先定义一个嵌入函数，然后将数据插入到集合中。

In [9]:
qwen_client=OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

def get_embeddings(texts:List[str])->List[List[float]]:
    if not texts:
        return []
    response=qwen_client.embeddings.create(
        model="qwen3-embedding:8b",
        input=texts,
    )
    return [item.embedding for item in response.data]

Insert example documents into the collection.

In [10]:
# documents=[
#     {
#         "content": "Milvus is a vector database built for embedding similarity search and AI applications.",
#         "metadata": {"source": "documentation", "topic": "introduction"},
#     },
#     {
#         "content": "Full-text search in Milvus allows you to search using keywords and phrases.",
#         "metadata": {"source": "tutorial", "topic": "full-text search"},
#     },
#     {
#         "content": "Hybrid search combines the power of sparse BM25 retrieval with dense vector search.",
#         "metadata": {"source": "blog", "topic": "hybrid search"},
#     },
# ]

from docx import Document
doc=Document('data/曾国藩知识.docx')

paragraphs=[p.text.strip() for p in doc.paragraphs if p.text.strip()]

documents=[]
for idx,para in enumerate(paragraphs):
    documents.append({
        "content": para,
        "metadata": {"source":"曾国藩知识.docx","paragraph_index":idx}
    })

entities=[]
texts=[doc['content'] for doc in documents]
embedings=get_embeddings(texts)

for i,doc in enumerate(documents):
    entities.append(
        {
            "content": doc['content'],
            "dense_vector": embedings[i],
            "metadata": doc.get('metadata', {}),
        }
    )

mc.insert(collection_name, entities)
print(f"Inserted {len(entities)} entities")

Inserted 42 entities


## 执行检索
您可以灵活使用 `search()` 或 `hybrid_search()` 方法，实现全文搜索（稀疏）、语义搜索（密集）以及混合搜索，从而获得更稳健、更准确的搜索结果。

### 全文搜索
稀疏搜索利用 BM25 算法查找包含特定关键词或短语的文档。这种传统搜索方法在精确匹配术语方面表现优异，尤其适用于用户明确知道所需内容的情况。

In [12]:
query_keyword='曾国藩谥号'

results = mc.search(
    collection_name=collection_name,
    data=[query_keyword],
    anns_field='sparse_vector',
    limit=5,
    output_fields=["content","metadata"],
)
sparse_results=results[0]

print("\nSparse Search (Full-text search):")
for i, result in enumerate(sparse_results):
    print(
        f"{i+1}. Score:{result['distance']}, Content:{result['entity']['content']}",
    )


Sparse Search (Full-text search):
1. Score:4.018857479095459, Content:曾国藩，男，初名曾子城，后改名曾国藩，初字居武，后改字伯涵，号涤生，谥号文正，一等毅勇侯，湖南省长沙府湘乡县（今湖南省湘乡县）人。生活于清朝，清嘉庆十六年（1811年）出生，清同治十一年二月初四(3月12日)（1872年）曾国藩病故于江南任所，享年62岁。
2. Score:0.6729167699813843, Content:1852年，曾国藩担任江西乡试正考官，途中得知母亲去世的讣告，曾国藩回家守孝。本应该守孝三年，但是才四个月，曾国藩收到了咸丰帝的圣旨，让其担任湖南帮办团练大臣。随后，曾国藩出山带兵，并且组建了一支非常有生气有战斗力的湘军。由于没有任何作战经验，曾国藩在战场上是屡战屡败。可是他屡败屡战。凭着打落牙齿活血吞这种精神和太平军对抗，1854年曾国藩兵败靖港，投水自杀，被下属救起。幸好，两天以后的湘潭大捷让朝廷没有追究曾国藩的靖港之败，让其继续带兵作战。1855年，曾国藩和石达开在江西交战，由于中了石达开的埋伏，损失了三分之二的水师，恰巧曾国藩得知父亲过世，随即从江西奔丧回老家建有思云馆，经过一年多的反思，曾国藩悟出了人生的很多道理，无人不拜无信不回，开始他人生的第二次出山带兵。
3. Score:0.6647834181785583, Content:1859年，湘军攻取安庆，这也标志着太平军在军事上的由盛转衰。第二年，朝廷授予曾国藩为两江总督，赏加兵部尚书衔，并以钦差大臣的身份督办江南军务。这是曾国藩出山带兵以来第一次担任实职，不到四年，湘军就攻取天京，太平军宣告失败。1864年，曾国藩也被朝廷追封为一等毅勇侯，汉人继吴三桂后朝廷是生不封公，死不封王，所以曾国藩封侯。很多人说曾国藩大智若愚，为什么呢？因为曾国藩立马主动裁军，一年多的时间，湘军裁撤只剩下两万多人，曾国藩的不居功自傲，不仅保全了曾氏家族，而且让湘军有一个善终。所以慈禧太后说曾国藩是晚清第一正人！
4. Score:0.656083881855011, Content:1868年，曾国藩调任直隶总督。兴办水利，大兴练军，开办洋务，一系列的措施让曾国藩赢得中兴第一名臣的美誉。无奈，1870年的天津教案使曾国藩陷入困境，还背负上卖国贼，汉奸的骂名。就

### 语义搜索
密集搜索利用向量嵌入来查找意义相似的文档，即使这些文档没有完全相同的关键词。这种方法有助于理解上下文和语义，因此非常适合处理更自然的语言查询。

In [13]:
query = "曾国藩的思想主张是什么？"

query_embeddings=get_embeddings([query])[0]

results=mc.search(
    collection_name=collection_name,
    data=[query_embeddings],
    anns_field='dense_vector',
    limit=5,
    output_fields=["content","metadata"],
)
dense_results=results[0]
print("\nDense Search (Semantic):")
for i, result in enumerate(dense_results):
    print(
        f"{i+1}. Score:{result['distance']:.4f}, Content:{result['entity']['content']}",
    )


Dense Search (Semantic):
1. Score:0.7016, Content:思想上，曾国藩一生奉行“程朱理学”，但对其并未盲目崇拜，对心学表现出了宽容的学术姿态，并以气学来弥补理学之局限，推动儒学的发展，时人称之为“圣相”。
2. Score:0.6783, Content:曾国藩知识内容
3. Score:0.6572, Content:曾国藩爱好书法，性格勤俭廉洁、意志力顽强、锐意功名，意气自豪、高度的“慎独”精神和强烈的使命感、谨言慎行、重贤重才，宽怀大度。
4. Score:0.6271, Content:这就是曾国藩的一生。
5. Score:0.6173, Content:曾国藩(1811—1872年)，字伯涵，号涤生，湖南湘乡人，是中国晚清重臣、军事家、政治家、文学家、战略家、理学家，是湘军的创立者和统帅、盐业改革者。清末汉族地主武装湘军的首领。道光进士，曾任内阁学士，道光末年官至侍郎。他这一生可分为四个阶段：读书科举时期、京师十年七迁时期、平定太平天国时期，吏治洋务时期。


### 混合搜索
混合搜索结合了全文搜索和语义密集检索，通过充分利用两种方法的优势，实现了搜索准确性和鲁棒性的平衡。
混合搜索在检索增强生成（RAG）应用中尤为有价值，因为语义理解与精确的关键词匹配共同提升了检索结果的质量。

In [14]:
query = "曾国藩当过哪些官"

query_embeddings=get_embeddings([query])[0]

sparse_search_params={'metric_type':'BM25'}
sparse_request=AnnSearchRequest(
    [query],
    'sparse_vector',
    sparse_search_params,
    limit=5,
)

dense_search_params = {"metric_type": "IP"}
dense_request=AnnSearchRequest(
    [query_embeddings],
    'dense_vector',
    dense_search_params,
    limit=5,
)

results=mc.hybrid_search(
    collection_name,
    [sparse_request,dense_request],
    ranker=RRFRanker(),
    limit=5,
    output_fields=["content","metadata"],
)
hybrid_results=results[0]

print("\nHybrid Search (Combined):")
for i, result in enumerate(hybrid_results):
    print(
        f"{i+1}. Score:{result['distance']:.4f}, Content:{result['entity']['content']}",
    )


Hybrid Search (Combined):
1. Score:0.0325, Content:30岁散馆考试，名列二等十九名，授翰林院检讨。33岁时钦命曾国藩为四川乡试正考官。同年8月，补授翰林院侍讲。同年12月，曾国藩充文渊阁校理。34岁，转侍读。35岁任会试同考官。同年5月，升詹事府右春坊右庶子。同年9月，转左庶子，不久升侍讲学士。同年12月，充日讲起居注官。36岁充文渊阁直阁事。37岁，大考二等。同年6月，升任内阁学士加礼部侍郎衔。38岁，稽察中书科事务。39岁授礼部右侍郎。同年8月，署兵部左侍郎。40岁署工部左侍郎。在京十多年间，曾国藩就是这样坚韧不拔地沿着这条仕途之道，步步升迁到二品官位。十年七迁，连跃十级。
2. Score:0.0164, Content:三弟曾国荃，军中都称之为“九帅”“九爷”，打仗勇猛，并且所任官职非常多，在后期也担任过两江总督，晚年实心任事，济民于困，被称为“赈灾专家”。
3. Score:0.0161, Content:1840年，30岁的曾国藩被授翰林院检讨，这是一个从七品的官职，主要从事一些抄抄写写的工作，从这里大家可以看到曾国藩的起点并不高，但是仅七年的时间，也就是曾国藩37岁的时候就升授内阁学士兼礼部侍郎，官至二品，七年连升十级，由此可见，曾国藩早期的仕途非常顺利。
4. Score:0.0159, Content:曾国藩墓位于长沙市今岳麓区坪塘街道桐溪寺后伏龙山上，清同治十三年（1874年）自河东金盆岭迁葬于此，占地面积300平方米。故居为富厚堂（位于湖南省娄底市双峰县荷叶镇富托村）。富厚堂建于1865年，其实始建是1857年，曾国藩父亲过世，他从江西奔丧回家，在山坡山建有思云馆，当时候这个宅院只是曾国荃的田庄，曾国藩居住期间认为这田庄乃一等好屋场，后期他认为最好的终老临泉之所就是这，由于存在产权的问题，和弟弟曾国荃商议，最后两兄弟兑换田庄，在家书里要求弟弟将思云馆修葺，没想到弟弟花费了7000串（相当于5000两白银）建富厚堂，曾国藩在家书得知后，责怪家人过于奢侈浪费，其实富厚堂的花费在于平整土地上，用料都非常的简单。5000两对于侯爷来说，只是半年的养廉费，可是对于秉承节俭家风的曾国藩来说这是一笔天大的数字。
5. Score:0.0159, Content:曾国藩(1811—1872年)，字伯涵，号涤生

## 答案生成
在通过混合搜索检索到相关文档后，我们可以使用大语言模型（LLM）根据所获取的信息生成全面的答案。这是RAG（检索增强生成）流程中的最后一步。

In [15]:
context="\n\n".join([doc['entity']['content'] for doc in hybrid_results])

prompt=f"""
根据提供的上下文回答以下问题。
如果上下文不包含相关信息，只需说“我无法回答这个问题，因为信息不足。”

上下文：{context}
问题：{query}

回答："""
deepseek_client=OpenAI(
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)
response=deepseek_client.chat.completions.create(
    model="deepseek-v4-pro",
    messages=[
        {"role":"system","content":"你是一位乐于助人的助手，会根据提供的上下文回答问题。"},
        {"role":"user","content":prompt},
    ]
)
print(response.choices[0].message.content)

根据上下文，曾国藩担任过的官职/职务主要包括：

翰林院检讨、四川乡试正考官、翰林院侍讲、文渊阁校理、侍读、会试同考官、詹事府右春坊右庶子、左庶子、侍讲学士、日讲起居注官、文渊阁直阁事、内阁学士加礼部侍郎衔、礼部右侍郎、署兵部左侍郎、署工部左侍郎等，官至二品。


In [ ]:
mc.drop_collection(collection_name)